# Multilingual Vision-Language Classification
Run this notebook from the extracted project directory. Read README.md for dataset scope, writer splits, architecture, and scientific limits. This notebook invokes the same tested Python modules; it contains no fabricated outputs.

In [ ]:
import sys, subprocess, os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
assert Path('pyproject.toml').exists(), 'Open this notebook from the extracted project root.'
def run(*args):
    subprocess.run([sys.executable, '-m', 'vlm.cli', *map(str,args)], check=True)


## Install dependencies
Use a compatible PyTorch CPU/CUDA build. This installs the project and its dataset helper in the current notebook environment.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[data,test]'], check=True)

## Download the real Chinese MNIST dataset
Kaggle access may require authentication. Alternatively, extract a manually downloaded archive and set RAW to its directory.

In [ ]:
import kagglehub
RAW = kagglehub.dataset_download('gpreda/chinese-mnist')
print(RAW)

In [ ]:
run('prepare', '--root', RAW, '--out', 'data/prepared', '--seed', 42)
TEXT = 'artifacts/text' if Path('artifacts/text/prototypes.npz').exists() else 'data/text'
if TEXT == 'data/text':
    run('text', '--out', TEXT)

## Train
Choose the settings before viewing held-out test performance. New output directories are required. Add --augment for the noise-trained comparison.

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
run('train', '--manifest', 'data/prepared/manifest.csv', '--prototypes', TEXT, '--out', 'runs/notebook', '--epochs', 30, '--device', DEVICE)

## Evaluate the selected checkpoint
This is held-out test evaluation, not the validation score used for checkpoint selection.

In [ ]:
run('evaluate', '--manifest', 'data/prepared/manifest.csv', '--checkpoint', 'runs/notebook/best.pt', '--out', 'runs/notebook_test', '--robust', '--device', DEVICE)

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv('runs/notebook_test/metrics.csv'))
display(Image(filename='runs/notebook/learning_curve.png'))
display(Image(filename='runs/notebook_test/robustness.png'))
display(Image(filename='runs/notebook_test/confusion_matrices.png'))

## Report measured numbers
Use the best epoch in history.csv for validation accuracy. Report separate per-language test accuracy and noisy accuracy from metrics.csv. Do not insert 96% unless an actual saved run supports it. Images contain Chinese handwriting; multilingual refers to the class prompts.